<a href="https://colab.research.google.com/github/lostysky/Data-Engineering-codeboosters-internship-2026/blob/main/Day_06_GenAI_PromptEngineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CELL 1
!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print('Libraries imported successfully')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00
Libraries imported successfully


In [ ]:
# This cell previously contained a hardcoded API key.
# It is recommended to use Colab's secrets manager for API keys.
# Please ensure your GROQ_API_KEY is set up in Colab's secrets and enabled for this notebook.

Groq Client configured with model: llama-3.1-8b-instant
Make sure API_KEY is replaced with you actual key!


In [ ]:
#CELL 3

def ask_llm(user_message, system_message='You are a helpful assistant.',temperature=0.7, max_tokens=500):
    """
    Send a message to the LLM and return the response text.

    Parameters:
    ----------------------
    user_message :
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            #system messages = instructions to the modek about how to behave
            #Like telling a new employee: 'You work at teh data company'
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        #temperature controls randomness
        #0.0 = always picks the most likely next token (consisten)
        max_tokens=max_tokens
        #maximum number od tokens in the response
        #500 tokens = 375 words
    )
    return response.choices[0].message.content
    #response.choices = list of possible completions (we requested 1)
    #[0]              = first completion
    #.message.content = the actual text the LLM generated

#Test the connection
test_response=ask_llm(
    "What is ETL in data engineering? Answer in exactly 2 sentences."
)
test_response2 = ask_llm(
    "Is GenAI abd Data Engineering a good career in 2026?"
)
print('=== LLM Response')
print(test_response)
print(test_response2)

=== LLM Response
ETL stands for Extract, Transform, Load, which is a standardized process in data engineering used to extract data from various sources, transform it into a standardized format, and load it into a destination system, such as a data warehouse or database. The ETL process is essential for integrating and analyzing data from multiple sources, ensuring data consistency, and enabling business intelligence and decision-making.
Based on current trends and forecasts, GenAI (General Artificial Intelligence) and Data Engineering are emerging as highly sought-after career paths in 2026. Here's why:

**GenAI:**

1. **Growing demand**: As AI technology advances, the demand for professionals who can develop and implement GenAI solutions is increasing rapidly.
2. **High-paying jobs**: GenAI specialists are in high demand and can command high salaries, with median salaries ranging from $150,000 to over $250,000 per year.
3. **Diverse applications**: GenAI has applications in various in

In [ ]:
response_etl = ask_llm(
    "In a 3 bullet points, explain how the Medallion Architecture "
    "(Bronze, Silver, Gold layer) related to ETL pipelines.",
    system_message="you are a senior data engineering instructor."
                   "Be concise and practical."
)
print('Medallion + ETL connection:')
print(response_etl)
print()
print("--- Token explanation ---")
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately', len(response_etl.split())*1.3, 'tokens.')
print('Llama-3.1-8b context window: 8192 tokens (~6000 words per conversation)')

Medallion + ETL connection:
Here are three key points explaining the Medallion Architecture in relation to ETL pipelines:

• **Bronze Layer**: This layer is used for **Ingestion and Storage**. It involves initial data collection from various sources into a raw, unprocessed format. Think of it as the first step in the ETL pipeline, where data is extracted from sources and loaded into a data warehouse or lake.

• **Silver Layer**: This layer is focused on **Data Quality and Transformation**. It involves applying data quality checks, data transformation, and data aggregation to the raw data from the Bronze layer. In the context of ETL, this would be the "T" in ETL (Extract, Transform, Load), where data is transformed into a more usable format.

• **Gold Layer**: This layer is dedicated to **Data Analysis, Reporting, and Visualization**. It involves applying advanced analytics, creating reports, and visualizing the transformed data from the Silver layer. In ETL terms, this would be the "L"

In [ ]:
zero_shot_response = ask_llm(
    "Extract the city name from this address:"
    "456 Bridge Road, Bangalore 560025, Karnataka, India"
)
print('Zero-Shot Result:')
print(zero_shot_response)

ambiguous_response =ask_llm("Clean this data: ramesh kumar,45000,mumbai")
print('Ambiguous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem: output format is unpredictable and not machine-parseable!')

Zero-Shot Result:
The city name is Bangalore.
Ambiguous Zero-Shot Result:
To clean this data, we need to identify the missing or incorrect information. Based on the given data: "ramesh kumar,45000,mumbai"

- The data seems to be a record of an employee's name, salary, and location. 
- The data is missing the employee's title, date of birth, or any other relevant information that might be needed.
- However, the provided information seems to be sufficient for basic record-keeping purposes.

To make the data more organized and clean, we can consider the following:

- Add a title to identify the type of record (e.g., "Employee Record:")
- Add a data format to the salary to make it clear (e.g., "Salary: 45,000 INR per annum")
- Consider adding a date format for record-keeping purposes (e.g., "Record Created: 01/01/2022")

Here's the cleaned data:

Employee Record:
Name: Ramesh Kumar
Salary: 45,000 INR per annum
Location: Mumbai
Record Created: (Insert Date)

If you want to further process t

In [ ]:
same_question = "Review this python code and identify any issues:\n" \
                "df['revenue'] = df['qty'] * df['price']\n" \
                "result = df.groupby('dept').sum()"

generic_response = ask_llm(same_question, temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300], '...')
print()


role_response = ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of production"
                   "experience. Review code critically for production readiness"
                   "data types issues, and potential failures at scale.",
    temperature=0.2
)

print('With Role Prompting (Senior Data ENgineer):')
print(role_response[:400], '...')
print()
print("Notice: role prompting produces more technical, actionable feedback")
#

Without Role Prompting:
The provided Python code appears to be a basic data manipulation task using the pandas library. However, there are a few potential issues that can be identified:

1. **Missing Error Handling**: The code does not include any error handling. If the 'qty' or 'price' columns do not exist in the DataFram ...

With Role Prompting (Senior Data ENgineer):
**Code Review**

The provided Python code appears to be a simple data manipulation task using the Pandas library. However, there are a few potential issues that could impact production readiness:

```python
# Assuming df is a DataFrame with columns 'qty', 'price', and 'dept'
df['revenue'] = df['qty'] * df['price']
result = df.groupby('dept').sum()
```

**Issues:**

1. **Data Types:**
   - The mult ...

Notice: role prompting produces more technical, actionable feedback


In [ ]:
prompt = "Give me one creative name for a data analytics startup."

print('=== Temperature Experiment ===')
for temp in [0.0, 0.5, 1.0]:
  response = ask_llm(prompt, temperature=temp)
  print(f'Temperature = {temp}: {response.strip()}')
  time.sleep(1)

print()
print('Observation:')
print('temperature=0.0 -> same or very similar answer every run (deterministic)')
print('temperature=0.5 -> some variation')
print('temperature=1.0 -> more creative/varied, sometimes surprising')
print()
print('Rule for data engineering tasks: use temperature=0.0 or 0.1')
print('You need CONSISTENT, PARSEABLE output - not creative variation')
#

=== Temperature Experiment ===
Temperature = 0.0: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature = 0.5: Here's a creative name for a data analytics startup: 

"Apexion Insights" 

This name suggests the idea of gaining deeper insights and maximizing potential, which aligns well with the goals of a data analytics startup.
Temperature = 1.0: One potential creative name for a data analytics startup could be "Kairos Insights".

The word "Kairos" is derived from ancient Greek, and it refers to the opportune moment or the perfect timing. This names suggests that the startup's data analytics solutions provide critical insights that help businesses make timely and informed decisions, capitalizing on 

In [ ]:
invoice_text = " Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024. Amount 45,000 for Laptop"

weak_response = ask_llm(
    f'Clean this invoice data: {invoice_text}',
    temperature=0.3
)
print('Weak Prompt Output:')
print(weak_response)
print()


try:
  json.loads(weak_response)
  print('PARSEABLE: Yes')
except:
  print('Parseable: No - cannot load into DataFrame')

print('\n' + '='*50 + '\n')

strong_system = """You are a data extraction specialist for an accounting pipeline.
Extract invoice data and return ONLY a valid JSON object.
DO NOT include any explanation, preamble, or markdown formatting.
Return ONLY the JSON, nothing else.

JSON schema (use null for missing values):
{"invoice_id": string, "vendor_name": string (Title Case),
"amount": number(no currency symbols),
"currency": string (default INr),
"invoice_date": string (YYYY-MM-DD),
"category": string (Electronics/Services/Accessories/Others)
}
"""

strong_response = ask_llm(
    f'Extract from: {invoice_text}',
    system_message=strong_system,
    temperature=0.0
)
print('Strong Prompt Output:')
print(strong_response)
print()
try:
  parsed = json.loads(strong_response.strip())
  print('PARSEABLE: Yes')
  print(f'Vendor: {parsed.get("vendor_name")}')
  print(f'Amount: {parsed.get("amount")}')
  print(f'Date: {parsed.get("invoice_date")}')
except json.JSONDecodeError:
  print('Parseable: No - cannot load into DataFrame')

Weak Prompt Output:
Here's the cleaned invoice data:

**Invoice Information:**

- **Invoice Number:** 2024-001
- **Invoice Date:** 15th January 2024
- **Vendor:** TECHWORLD SOLUTIONS

**Invoice Details:**

- **Item:** Laptop
- **Amount:** $45,000

I've reformatted the date to a more standard format and added a clear label for the vendor. I've also assumed the amount is in USD, but you may need to adjust this based on the actual currency used.

Parseable: No - cannot load into DataFrame


Strong Prompt Output:
{"invoice_id": "2024-001", "vendor_name": "Techworld Solutions", "amount": 45000, "currency": "INR", "invoice_date": "2024-01-15", "category": "Electronics"}

PARSEABLE: Yes
Vendor: Techworld Solutions
Amount: 45000
Date: 2024-01-15


# mini project:smart data cleaner

In [ ]:
few_shot_prompt_short = '''
Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:
'''
print('Short Few-Shot Prompt:')
print(few_shot_prompt_short)
# Expected output for 'ALICE JOHNSON, 82000' would be: {"name": "Alice Johnson", "salary": 82000}

Short Few-Shot Prompt:

Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:



In [ ]:
import json
import re

llm_output_bad = '```json\n{"name":"Ramesh"}\n```'

# Use regex to extract the pure JSON string
match = re.search(r'```json\n(.*)\n```', llm_output_bad, re.DOTALL)
if match:
    json_string_fixed = match.group(1).strip()
    try:
        parsed_data = json.loads(json_string_fixed)
        print(f"Fixed: Successfully parsed as: {parsed_data}")
    except json.JSONDecodeError as e:
        print(f"Error even after regex: {e}")
else:
    print("No JSON block found.")

Fixed: Successfully parsed as: {'name': 'Ramesh'}


In [ ]:
# Make sure to run all preceding cells, especially the one defining `ask_llm`.
few_shot_prompt = """
Convert employee text to JSON. Here are examples:

Input: RAMESH KUMAR, 45000, mumbai
Output: {"name": "Ramesh Kumar", "salary": 45000, "city": "Mumbai"}

Input: priya nair, 52000, Delhi
Output: {"name": "Priya Nair", "salary": 52000, "city": "Delhi"}

Now convert this:
Input: ANANYA DAS, 38000, kolkata
Output:
"""
few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()

try:
  # The type hint 'parsed: json.loads' is incorrect. It should be 'parsed = json.loads'
  parsed = json.loads(few_shot_response.strip())
  print('Sucessfully parsed as JSON!')
  print(f'Name: {parsed["name"]}, Salary: {parsed["salary"]}, City: {parsed["city"]}')

except json.JSONDecodeError:
  print('Parsing failed - model added extra text or invalid JSON.')
  print("Solution: add explicit instructions in the system prompt to return ONLY JSON, and check your prompt examples for correct JSON syntax.")


Few-Shot Result:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(', ')
    
    # Create a dictionary with the given keys
    employee_data = {
        "name": values[0].strip().title(),
        "salary": int(values[1]),
        "city": values[2].strip().title()
    }
    
    # Convert the dictionary to JSON
    json_data = json.dumps(employee_data)
    
    return json_data

# Test the function
employee_text = "ANANYA DAS, 38000, kolkata"
print(convert_to_json(employee_text))
```

When you run this function with the input "ANANYA DAS, 38000, kolkata", it will output:

```json
{"name": "Ananya Das", "salary": 38000, "city": "Kolkata"}
```

This function works by splitting the input string into individual values using the `split(', ')` method. It then creates a dictionary with the given keys and assigns the corresponding v